In [ ]:
#!/usr/bin/env python3
"""Continue a MISMIP+ Icepack run from a spin-up checkpoint.
"""

import argparse
import json
import os
from pathlib import Path
import time as walltime

import numpy as np
import tqdm

import firedrake
from firedrake import (
    Constant,
    Function,
    PointNotInDomainError,
    conditional,
    dx,
    inner,
    max_value,
    sin,
    sqrt,
    tanh,
)
from firedrake.exceptions import ConvergenceError

import icepack
from icepack.constants import (
    gravity as g,
    ice_density as rho_i,
    water_density as rho_w,
    weertman_sliding_law as m,
)
from icepack.models.viscosity import viscosity_depth_averaged
from mpi4py import MPI


MUTABLE_CFG_KEYS = {
    "A",
    "C",
    "a",
    "Ax",
    "Ay",
    "omega",
    "grid_resolution",
    "monitor_snes",
    "step_print_every",
}
A_OVERRIDE_KEYS = {"A", "Ax", "Ay"}


def print0(comm, message):
    """Print once in MPI runs."""
    if comm.rank == 0:
        print(message, flush=True)


def to_jsonable(value):
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    return value


def read_json_file(filename):
    with open(filename, "r", encoding="utf-8") as stream:
        value = json.load(stream)
    if not isinstance(value, dict):
        raise ValueError(f"Expected a JSON object in {filename!r}.")
    return value


def _decode_hdf5_string(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    if hasattr(value, "item"):
        value = value.item()
        if isinstance(value, bytes):
            return value.decode("utf-8")
    return str(value)


def load_spinup_cfg(checkpoint_filename, cfg_filename=None):
    """Load the cfg paired with a spin-up checkpoint.
    """
    checkpoint_filename = os.fspath(checkpoint_filename)

    if cfg_filename is not None:
        return read_json_file(cfg_filename), os.fspath(cfg_filename)

    embedded_cfg_filename = None
    try:
        import h5py

        with h5py.File(checkpoint_filename, "r") as h5:
            if "cfg_json" in h5.attrs:
                raw = _decode_hdf5_string(h5.attrs["cfg_json"])
                cfg = json.loads(raw)
                if not isinstance(cfg, dict):
                    raise ValueError("Embedded cfg_json is not a JSON object.")
                return cfg, f"{checkpoint_filename}:cfg_json"

            if "cfg_filename" in h5.attrs:
                embedded_cfg_filename = _decode_hdf5_string(
                    h5.attrs["cfg_filename"]
                )
    except (ImportError, OSError, ValueError, json.JSONDecodeError):
        pass

    candidates = []
    checkpoint_path = Path(checkpoint_filename)
    if embedded_cfg_filename:
        candidates.append(checkpoint_path.parent / embedded_cfg_filename)
    candidates.append(checkpoint_path.with_suffix(".json"))

    for candidate in candidates:
        if candidate.is_file():
            return read_json_file(candidate), os.fspath(candidate)

    raise FileNotFoundError(
        "Could not find the spin-up cfg. Supply --cfg-file/--fname, keep the "
        "sidecar JSON next to the checkpoint, or use a checkpoint containing "
        "the cfg_json HDF5 attribute written by spinup_new_full.py."
    )


def parse_override_assignment(text):
    """Parse KEY=VALUE, using JSON syntax for VALUE when possible."""
    if "=" not in text:
        raise argparse.ArgumentTypeError(
            f"Invalid override {text!r}; expected KEY=VALUE."
        )

    key, raw_value = text.split("=", 1)
    key = key.strip()
    raw_value = raw_value.strip()
    if not key:
        raise argparse.ArgumentTypeError("Override key cannot be empty.")

    try:
        value = json.loads(raw_value)
    except json.JSONDecodeError:
        value = raw_value

    return key, value


def validate_override_keys(overrides, source):
    unknown = sorted(set(overrides) - MUTABLE_CFG_KEYS)
    if unknown:
        allowed = ", ".join(sorted(MUTABLE_CFG_KEYS))
        raise ValueError(
            f"Unsupported forward-run cfg key(s) in {source}: "
            f"{', '.join(unknown)}. Allowed keys: {allowed}. Geometry and mesh "
            "settings cannot be changed after loading a checkpoint."
        )


def merge_cfg(spinup_cfg, override_layers):
    """Apply ordered partial override dictionaries.
    """
    cfg = dict(spinup_cfg)
    explicit_keys = set()
    changes = []

    for source, overrides in override_layers:
        if not overrides:
            continue
        validate_override_keys(overrides, source)
        for key, value in overrides.items():
            old_value = cfg.get(key, "<missing>")
            cfg[key] = value
            explicit_keys.add(key)
            changes.append(
                {
                    "source": source,
                    "key": key,
                    "old": to_jsonable(old_value),
                    "new": to_jsonable(value),
                }
            )

    return cfg, explicit_keys, changes


def global_mesh_bounds(mesh):
    coordinates = mesh.coordinates.dat.data_ro
    local_min = coordinates.min(axis=0)
    local_max = coordinates.max(axis=0)
    comm = mesh.comm

    global_min = np.array(
        [comm.allreduce(float(value), op=MPI.MIN) for value in local_min]
    )
    global_max = np.array(
        [comm.allreduce(float(value), op=MPI.MAX) for value in local_max]
    )
    return global_min, global_max


def validate_cfg_and_mesh(cfg, mesh):
    for key in ("C", "a"):
        if key not in cfg:
            raise KeyError(
                f"The spin-up cfg has no {key!r}. Supply it explicitly with "
                f"--{key} or --set {key}=VALUE."
            )

    C_value = float(cfg["C"])
    if not np.isfinite(C_value) or C_value <= 0.0:
        raise ValueError(f"C must be finite and positive; got {cfg['C']!r}.")

    a_value = float(cfg["a"])
    if not np.isfinite(a_value):
        raise ValueError(f"a must be finite; got {cfg['a']!r}.")

    omega = float(cfg.get("omega", 0.0))
    if not np.isfinite(omega):
        raise ValueError(f"omega must be finite; got {omega!r}.")

    resolution = float(cfg.get("grid_resolution", 500.0))
    if not np.isfinite(resolution) or resolution <= 0.0:
        raise ValueError(
            "grid_resolution must be finite and positive; "
            f"got {resolution!r}."
        )

    xyz_min, xyz_max = global_mesh_bounds(mesh)
    span = xyz_max - xyz_min

    for index, key in enumerate(("Lx", "Ly")):
        if key not in cfg:
            cfg[key] = float(span[index])
            continue

        configured = float(cfg[key])
        tolerance = max(1.0e-8 * abs(configured), 1.0e-6)
        if abs(configured - float(span[index])) > tolerance:
            raise ValueError(
                f"The cfg value {key}={configured} does not match the loaded "
                f"mesh span {span[index]}. The JSON may not belong to this "
                "checkpoint."
            )


def friction(**kwargs):
    """Regularized Coulomb/Weertman-style basal friction used in spin-up."""
    u, h, s, C = map(
        kwargs.get, ("velocity", "thickness", "surface", "friction")
    )

    p_w = rho_w * g * max_value(0, -(s - h))
    p_i = rho_i * g * h
    effective_pressure = max_value(0, p_i - p_w)
    tau_c = effective_pressure / 2

    u_c = (tau_c / C) ** m
    u_b = sqrt(inner(u, u))

    phi = tau_c * (
        (u_c**(1 / m + 1) + u_b**(1 / m + 1))**(m / (m + 1)) - u_c
    )
    sliding_scale = Constant(1e0)   

    return sliding_scale * phi


def viscosity(**kwargs):
    return viscosity_depth_averaged(
        velocity=kwargs["velocity"],
        thickness=kwargs["thickness"],
        fluidity=kwargs["fluidity"],
    )


def firedrake_smooth(q0, alpha=2.0e3):
    q = q0.copy(deepcopy=True)
    objective = 0.5 * (
        (q - q0) ** 2
        + alpha**2 * firedrake.inner(firedrake.grad(q), firedrake.grad(q))
    ) * firedrake.dx
    residual = firedrake.derivative(objective, q)
    firedrake.solve(residual == 0, q)
    return q


def flotation_height(bed, Q):
    return Function(Q).interpolate(
        max_value(-bed * (rho_w / rho_i - 1), 0)
    )


def flotation_mask(surface, z_flotation, Q):
    z_above = firedrake_smooth(
        Function(Q).interpolate(surface - z_flotation), alpha=100.0
    )
    floating = Function(Q).interpolate(
        conditional(z_above < 0, 1.0, 0.0)
    )
    return floating


def mismip_melt_rate(omega, surface, thickness, bed, Q):
    """Return accumulation contribution from the legacy MISMIP+ melt law."""
    z_flotation = flotation_height(bed, Q)
    floating = flotation_mask(surface, z_flotation, Q)
    ice_base = (surface - thickness) * floating
    cavity_thickness = (ice_base - bed) * floating

    Hc0 = Constant(75.0)
    z0 = Constant(-100.0)
    melt = Constant(omega) * tanh(cavity_thickness / Hc0) * max_value(
        z0 - ice_base, 0
    )
    return -melt


def make_A_field(mesh, Q, checkpoint_A, cfg, explicit_keys):
    """Choose the exact saved A field unless A-related values were overridden."""
    if not (A_OVERRIDE_KEYS & explicit_keys):
        if checkpoint_A is None:
            if "A" not in cfg:
                raise KeyError(
                    "The checkpoint has no saved A field and the cfg has no A. "
                    "Supply --A."
                )
            A_field = Function(Q, name="A").interpolate(Constant(float(cfg["A"])))
            return A_field, "cfg-constant-fallback"

        A_field = Function(Q, name="A")
        A_field.project(checkpoint_A)
        return A_field, "checkpoint"

    if "A" not in cfg:
        raise KeyError("An A/Ax/Ay override requires a base value A.")

    A_base = float(cfg["A"])
    Ax = float(cfg.get("Ax", 0.0))
    Ay = float(cfg.get("Ay", 0.0))
    for name, value in (("A", A_base), ("Ax", Ax), ("Ay", Ay)):
        if not np.isfinite(value):
            raise ValueError(f"{name} must be finite; got {value!r}.")

    # On this MISMIP+ rectangle, sin(pi*x/Lx) and sin(pi*y/Ly) are in [0, 1].
    lower_bound = A_base + min(0.0, Ax) + min(0.0, Ay)
    if lower_bound <= 0.0:
        raise ValueError(
            "The requested A field is non-positive somewhere: "
            f"A + min(0, Ax) + min(0, Ay) = {lower_bound}."
        )

    x, y = firedrake.SpatialCoordinate(mesh)
    expression = (
        Constant(A_base)
        + Constant(Ax) * sin(np.pi * x / float(cfg["Lx"]))
        + Constant(Ay) * sin(np.pi * y / float(cfg["Ly"]))
    )
    A_field = Function(Q, name="A").interpolate(expression)

    if Ax == 0.0 and Ay == 0.0:
        return A_field, "overridden-constant"
    return A_field, "overridden-sinusoidal"


def make_solver(kind, model, monitor=False):
    base_options = {
        "dirichlet_ids": [4],
        "side_wall_ids": [1, 3],
    }

    if kind == "icepack":
        options = {
            **base_options,
            "diagnostic_solver_type": "icepack",
            "diagnostic_solver_parameters": {
                "ksp_type": "preonly",
                "pc_type": "lu",
                "pc_factor_mat_solver_type": "mumps",
                "tolerance": 1.0e-8,
                "max_iterations": 200,
            },
        }
    elif kind == "petsc":
        diagnostic_parameters = {
            "snes_type": "newtonls",
            "snes_linesearch_type": "cp",
            "snes_max_it": 200,
            "ksp_type": "preonly",
            "pc_type": "lu",
            "pc_factor_mat_solver_type": "mumps",
        }
        if monitor:
            diagnostic_parameters.update(
                {
                    "snes_monitor": None,
                    "snes_converged_reason": None,
                    "ksp_converged_reason": None,
                }
            )
        options = {
            **base_options,
            "diagnostic_solver_type": "petsc",
            "diagnostic_solver_parameters": diagnostic_parameters,
        }
    else:
        raise ValueError(f"Unknown solver kind {kind!r}.")

    return icepack.solvers.FlowSolver(model, **options)


def diagnostic_solve_with_fallback(
    primary_solver,
    fallback_solver,
    *,
    velocity,
    thickness,
    surface,
    fluidity,
    friction_coefficient,
    stage_name,
):
    velocity_before = velocity.copy(deepcopy=True)
    try:
        return primary_solver.diagnostic_solve(
            velocity=velocity,
            thickness=thickness,
            surface=surface,
            fluidity=fluidity,
            friction=friction_coefficient,
        )
    except ConvergenceError as error:
        if fallback_solver is None:
            raise

        print0(
            thickness.function_space().mesh().comm,
            f"{stage_name}: PETSc diagnostic solve failed; retrying with "
            f"Icepack's solver. Original error: {error!r}",
        )
        return fallback_solver.diagnostic_solve(
            velocity=velocity_before,
            thickness=thickness,
            surface=surface,
            fluidity=fluidity,
            friction=friction_coefficient,
        )


def make_time_steps(duration, dt):
    if not np.isfinite(duration) or duration < 0.0:
        raise ValueError(f"final_time must be finite and non-negative; got {duration}.")
    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(f"dt must be finite and positive; got {dt}.")
    if duration == 0.0:
        return []

    n_full = int(np.floor(duration / dt + 1.0e-12))
    steps = [float(dt)] * n_full
    remainder = float(duration - n_full * dt)
    tolerance = 1.0e-12 * max(1.0, duration, dt)
    if remainder > tolerance:
        steps.append(remainder)
    elif remainder < -tolerance:
        raise RuntimeError("Internal time-step construction error.")
    return steps


def run_simulation(
    primary_solver,
    fallback_solver,
    fields,
    *,
    bed,
    A_field,
    C_value,
    accumulation_value,
    omega,
    duration,
    dt,
    print_every,
):
    h = fields["thickness"]
    s = fields["surface"]
    u = fields["velocity"]
    mesh = h.function_space().mesh()
    Q = h.function_space()
    comm = mesh.comm

    C = Constant(C_value)
    accumulation = Constant(accumulation_value)
    thickness_inflow = h.copy(deepcopy=True)
    domain_area = float(firedrake.assemble(Constant(1.0) * dx(domain=mesh)))
    time_steps = make_time_steps(duration, dt)

    u = diagnostic_solve_with_fallback(
        primary_solver,
        fallback_solver,
        velocity=u,
        thickness=h,
        surface=s,
        fluidity=A_field,
        friction_coefficient=C,
        stage_name="initial state",
    )

    initial_fields = {
        "thickness": h.copy(deepcopy=True),
        "surface": s.copy(deepcopy=True),
        "velocity": u.copy(deepcopy=True),
    }

    progress = tqdm.tqdm(
        enumerate(time_steps, start=1),
        total=len(time_steps),
        desc="forward run",
        disable=(comm.rank != 0),
    )

    elapsed = 0.0
    for step_index, step_dt in progress:
        t0 = walltime.time()

        if float(omega) == 0.0:
            net_accumulation = accumulation
        else:
            melt = mismip_melt_rate(omega, s, h, bed, Q)
            net_accumulation = Function(Q).interpolate(melt + accumulation)

        h = primary_solver.prognostic_solve(
            step_dt,
            thickness=h,
            velocity=u,
            accumulation=net_accumulation,
            thickness_inflow=thickness_inflow,
        )

        # Match the spin-up admissibility safeguard.
        h.interpolate(max_value(h, 1.0))
        s = icepack.compute_surface(thickness=h, bed=bed)

        u = diagnostic_solve_with_fallback(
            primary_solver,
            fallback_solver,
            velocity=u,
            thickness=h,
            surface=s,
            fluidity=A_field,
            friction_coefficient=C,
            stage_name=f"step {step_index}",
        )

        elapsed += step_dt
        local_min_h = float(h.dat.data_ro.min())
        minimum_h = comm.allreduce(local_min_h, op=MPI.MIN)
        average_h = float(firedrake.assemble(h * dx) / domain_area)
        progress.set_description(
            f"t={elapsed:g} yr; avg,min h={average_h:.2f},{minimum_h:.2f}"
        )

        if print_every and (
            step_index == 1
            or step_index % int(print_every) == 0
            or step_index == len(time_steps)
        ):
            print0(
                comm,
                f"step {step_index}/{len(time_steps)}, t={elapsed:g} yr: "
                f"avg h={average_h:.3f}, min h={minimum_h:.3f}, "
                f"wall={walltime.time() - t0:.2f} s",
            )

    final_fields = {"thickness": h, "surface": s, "velocity": u}
    return initial_fields, final_fields, time_steps


def load_checkpoint_fields(checkpoint_filename):
    with firedrake.CheckpointFile(checkpoint_filename, "r") as checkpoint:
        mesh = checkpoint.load_mesh()
        velocity = checkpoint.load_function(mesh, "velocity")
        thickness = checkpoint.load_function(mesh, "thickness")
        surface = checkpoint.load_function(mesh, "surface")
        bed = checkpoint.load_function(mesh, "bed")
        try:
            A_field = checkpoint.load_function(mesh, "A")
        except Exception as error:  # Compatibility fallback for older checkpoints.
            print0(
                mesh.comm,
                f"Warning: checkpoint A field could not be loaded ({error!r}); "
                "the cfg A value will be used if available.",
            )
            A_field = None

    Q = thickness.function_space()
    V = velocity.function_space()

    h = Function(Q, name="h").project(thickness)
    b = Function(Q, name="b").project(bed)
    s = Function(Q, name="s").project(surface)
    u = Function(V, name="u").project(velocity)

    reconstructed_surface = icepack.compute_surface(thickness=h, bed=b)
    try:
        surface_difference = Function(Q).interpolate(abs(s - reconstructed_surface))
        local_max = float(surface_difference.dat.data_ro.max())
        max_difference = mesh.comm.allreduce(local_max, op=MPI.MAX)
        if max_difference > 1.0e-8:
            print0(
                mesh.comm,
                "Warning: saved surface differs from compute_surface(h, bed) by "
                f"up to {max_difference:.6g} m; using the reconstructed surface.",
            )
    except Exception:
        pass
    s = reconstructed_surface

    return mesh, Q, V, {"thickness": h, "surface": s, "velocity": u}, b, A_field


def write_json(cfg, filename, comm):
    if filename is None:
        return
    if comm.rank == 0:
        path = Path(filename)
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("w", encoding="utf-8") as stream:
            json.dump(to_jsonable(cfg), stream, indent=2, sort_keys=True)
    comm.barrier()


def embed_cfg_in_checkpoint(checkpoint_filename, cfg, cfg_filename, comm):
    comm.barrier()
    if comm.rank == 0:
        try:
            import h5py

            with h5py.File(checkpoint_filename, "a") as h5:
                h5.attrs["cfg_json"] = json.dumps(
                    to_jsonable(cfg), sort_keys=True
                )
                if cfg_filename is not None:
                    h5.attrs["cfg_filename"] = os.path.basename(cfg_filename)
        except Exception as error:
            print(
                f"Warning: could not embed cfg in {checkpoint_filename}: "
                f"{error!r}",
                flush=True,
            )
    comm.barrier()


def save_checkpoint(
    filename,
    mesh,
    Q,
    V,
    fields,
    bed,
    A_field,
    cfg,
    cfg_filename,
):
    comm = mesh.comm
    path = Path(filename)
    if comm.rank == 0:
        path.parent.mkdir(parents=True, exist_ok=True)
    comm.barrier()

    u = Function(V, name="u").project(fields["velocity"])
    h = Function(Q, name="h").project(fields["thickness"])
    s = Function(Q, name="s").project(fields["surface"])
    b = Function(Q, name="b").project(bed)
    A_out = Function(Q, name="A").project(A_field)

    with firedrake.CheckpointFile(os.fspath(path), "w") as checkpoint:
        checkpoint.save_mesh(mesh)
        checkpoint.save_function(u, name="velocity")
        checkpoint.save_function(h, name="thickness")
        checkpoint.save_function(s, name="surface")
        checkpoint.save_function(b, name="bed")
        checkpoint.save_function(A_out, name="A")

    embed_cfg_in_checkpoint(os.fspath(path), cfg, cfg_filename, comm)
    print0(comm, f"Saved continuation checkpoint: {path}")


def evaluate_function_on_grid(function, points, values, block=50_000):
    count = points.shape[0]
    for start in range(0, count, block):
        stop = min(start + block, count)
        point_block = points[start:stop]
        try:
            values[start:stop] = function.at(point_block)
        except PointNotInDomainError:
            for offset, point in enumerate(point_block):
                try:
                    values[start + offset] = function.at(point)
                except PointNotInDomainError:
                    pass
    return values


def _evaluate_state_on_grid(fields, points):
    h = fields["thickness"]
    s = fields["surface"]
    u = fields["velocity"]
    count = points.shape[0]

    h_values = np.full(count, np.nan, dtype=float)
    s_values = np.full(count, np.nan, dtype=float)
    u_values = np.full((count, 2), np.nan, dtype=float)
    evaluate_function_on_grid(h, points, h_values)
    evaluate_function_on_grid(s, points, s_values)
    evaluate_function_on_grid(u, points, u_values)
    return h_values, s_values, u_values


def save_grid_npz(
    filename,
    mesh,
    Q,
    initial_fields,
    final_fields,
    bed,
    A_field,
    cfg,
):
    comm = mesh.comm
    if comm.size != 1:
        raise RuntimeError(
            "Regular-grid NPZ export currently requires one MPI rank. Run "
            "serially or pass --no-grid; checkpoint output remains parallel."
        )

    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)

    bounds_min, bounds_max = global_mesh_bounds(mesh)
    xmin, ymin = bounds_min
    xmax, ymax = bounds_max
    resolution = float(cfg.get("grid_resolution", 500.0))

    x = np.arange(xmin, xmax + 0.5 * resolution, resolution)
    y = np.arange(ymin, ymax + 0.5 * resolution, resolution)
    x = x[x <= xmax + 1.0e-8]
    y = y[y <= ymax + 1.0e-8]
    if len(x) == 0 or len(y) == 0:
        raise RuntimeError("The regular output grid is empty.")

    X, Y = np.meshgrid(x, y, indexing="xy")
    points = np.column_stack([X.ravel(), Y.ravel()])
    count = points.shape[0]

    h0_values, s0_values, u0_values = _evaluate_state_on_grid(
        initial_fields, points
    )
    h_values, s_values, u_values = _evaluate_state_on_grid(final_fields, points)

    bed_values = np.full(count, np.nan, dtype=float)
    A_values = np.full(count, np.nan, dtype=float)
    speed_values = np.full(count, np.nan, dtype=float)
    haf_values = np.full(count, np.nan, dtype=float)
    viscosity_values = np.full(count, np.nan, dtype=float)

    h = final_fields["thickness"]
    s = final_fields["surface"]
    u = final_fields["velocity"]
    speed = Function(Q, name="speed").interpolate(sqrt(inner(u, u)))
    height_above_flotation = Function(
        Q, name="height_above_flotation"
    ).interpolate(s - (1 - rho_i / rho_w) * h)
    effective_viscosity = firedrake.project(
        viscosity(velocity=u, thickness=h, fluidity=A_field), Q
    )

    evaluate_function_on_grid(bed, points, bed_values)
    evaluate_function_on_grid(A_field, points, A_values)
    evaluate_function_on_grid(speed, points, speed_values)
    evaluate_function_on_grid(
        height_above_flotation, points, haf_values
    )
    evaluate_function_on_grid(effective_viscosity, points, viscosity_values)

    ny, nx = len(y), len(x)
    initial_velocity = u0_values.reshape(ny, nx, 2)
    final_velocity = u_values.reshape(ny, nx, 2)

    np.savez_compressed(
        path,
        x=x,
        y=y,
        X=X,
        Y=Y,
        h=h_values.reshape(ny, nx),
        thickness=h_values.reshape(ny, nx),
        s=s_values.reshape(ny, nx),
        surface=s_values.reshape(ny, nx),
        bed=bed_values.reshape(ny, nx),
        ux=final_velocity[..., 0],
        uy=final_velocity[..., 1],
        velocity=final_velocity,
        speed=speed_values.reshape(ny, nx),
        A=A_values.reshape(ny, nx),
        A_inv=A_values.reshape(ny, nx),
        viscosity=viscosity_values.reshape(ny, nx),
        height_above_flotation=haf_values.reshape(ny, nx),
        initial_h=h0_values.reshape(ny, nx),
        initial_thickness=h0_values.reshape(ny, nx),
        initial_s=s0_values.reshape(ny, nx),
        initial_surface=s0_values.reshape(ny, nx),
        initial_ux=initial_velocity[..., 0],
        initial_uy=initial_velocity[..., 1],
        initial_velocity=initial_velocity,
        xmin=float(xmin),
        xmax=float(xmax),
        ymin=float(ymin),
        ymax=float(ymax),
        grid_resolution=resolution,
        cfg_json=json.dumps(to_jsonable(cfg), sort_keys=True),
    )
    print0(
        comm,
        f"Saved regular-grid NPZ: {path} with shape (ny={ny}, nx={nx})",
    )


def derive_output_paths(args):
    checkpoint_path = Path(args.ss_filename)
    if args.output_stem is None:
        time_label = f"{args.final_time:g}".replace(".", "p")
        stem = checkpoint_path.with_suffix("").with_name(
            f"{checkpoint_path.stem}_forward_{time_label}yr"
        )
    else:
        stem = Path(args.output_stem)
        if stem.suffix in {".h5", ".hdf5", ".json", ".npz"}:
            stem = stem.with_suffix("")

    output_checkpoint = (
        Path(args.output_checkpoint)
        if args.output_checkpoint is not None
        else Path(f"{stem}.h5")
    )
    output_config = (
        Path(args.output_config)
        if args.output_config is not None
        else Path(f"{stem}.json")
    )
    grid_output = (
        Path(args.savename)
        if args.savename is not None
        else Path(f"{stem}_grid.npz")
    )

    if not args.no_checkpoint:
        try:
            same_file = output_checkpoint.resolve() == checkpoint_path.resolve()
        except FileNotFoundError:
            same_file = os.path.abspath(output_checkpoint) == os.path.abspath(
                checkpoint_path
            )
        if same_file:
            raise ValueError(
                "The output checkpoint must differ from the input checkpoint."
            )

    return output_checkpoint, output_config, grid_output


def make_parser():
    parser = argparse.ArgumentParser(
        description=(
            "Continue a MISMIP+ Icepack simulation from the steady-state "
            "checkpoint written by spinup_new_full.py."
        )
    )
    parser.add_argument(
        "--ss-filename",
        "--ss_filename",
        required=True,
        help="Input steady-state HDF5 checkpoint.",
    )
    parser.add_argument(
        "--cfg-file",
        "--fname",
        default=None,
        help=(
            "Spin-up JSON cfg. By default it is read from cfg_json embedded in "
            "the checkpoint, then from a same-stem sidecar JSON."
        ),
    )
    parser.add_argument(
        "--override-json",
        default=None,
        help=(
            "Partial JSON object containing only forward-run overrides. "
            "Supported keys: A, C, a, Ax, Ay, omega, grid_resolution, "
            "monitor_snes, step_print_every."
        ),
    )
    parser.add_argument(
        "--set",
        dest="set_overrides",
        action="append",
        default=[],
        metavar="KEY=VALUE",
        help="Set a forward cfg value; may be repeated.",
    )

    parser.add_argument(
        "--final-time",
        "--final_time",
        type=float,
        default=1.0,
        help="Forward-run duration in years (default: 1).",
    )
    parser.add_argument(
        "--dt",
        type=float,
        default=None,
        help=(
            "Time step in years. Default: run_dt from the input cfg, then "
            "fine_dt, then coarse_dt, then 1 year."
        ),
    )

    parser.add_argument("--C", type=float, default=None, help="Sliding coefficient.")
    parser.add_argument("--A", type=float, default=None, help="Base fluidity.")
    parser.add_argument("--a", type=float, default=None, help="Accumulation (m/yr).")
    parser.add_argument(
        "--Ax",
        type=float,
        default=None,
        help="Amplitude of sin(pi*x/Lx) added to base A.",
    )
    parser.add_argument(
        "--Ay",
        type=float,
        default=None,
        help="Amplitude of sin(pi*y/Ly) added to base A.",
    )
    parser.add_argument(
        "--omega",
        type=float,
        default=None,
        help="Legacy basal-melt scaling; default is inherited or 0.",
    )
    parser.add_argument(
        "--grid-resolution",
        dest="grid_resolution",
        type=float,
        default=None,
        help="Regular-grid NPZ spacing in metres.",
    )

    parser.add_argument(
        "--output-stem",
        default=None,
        help=(
            "Output path without suffix. Defaults to an input-stem-derived "
            "name. Produces .h5, .json, and _grid.npz."
        ),
    )
    parser.add_argument(
        "--output-checkpoint",
        default=None,
        help="Override the continuation HDF5 output path.",
    )
    parser.add_argument(
        "--output-config",
        default=None,
        help="Override the effective JSON cfg output path.",
    )
    parser.add_argument(
        "--savename",
        default=None,
        help="Backward-compatible alias for the regular-grid NPZ output path.",
    )
    parser.add_argument(
        "--no-checkpoint",
        action="store_true",
        help="Do not write a restartable final HDF5 checkpoint.",
    )
    parser.add_argument(
        "--no-grid",
        action="store_true",
        help="Do not write the regular-grid NPZ output.",
    )
    return parser


def main(argv=None):
    parser = make_parser()
    args = parser.parse_args(argv)

    spinup_cfg, cfg_source = load_spinup_cfg(args.ss_filename, args.cfg_file)

    override_layers = []
    if args.override_json is not None:
        override_layers.append(
            (args.override_json, read_json_file(args.override_json))
        )

    set_overrides = {}
    for item in args.set_overrides:
        key, value = parse_override_assignment(item)
        set_overrides[key] = value
    override_layers.append(("--set", set_overrides))

    dedicated_overrides = {
        key: value
        for key, value in {
            "C": args.C,
            "A": args.A,
            "a": args.a,
            "Ax": args.Ax,
            "Ay": args.Ay,
            "omega": args.omega,
            "grid_resolution": args.grid_resolution,
        }.items()
        if value is not None
    }
    override_layers.append(("dedicated CLI flags", dedicated_overrides))

    cfg, explicit_keys, changes = merge_cfg(spinup_cfg, override_layers)
    cfg.setdefault("omega", 0.0)
    cfg.setdefault("Ax", 0.0)
    cfg.setdefault("Ay", 0.0)
    cfg.setdefault("grid_resolution", 500.0)
    cfg.setdefault("monitor_snes", False)
    cfg.setdefault("step_print_every", 10)

    mesh, Q, V, fields, bed, checkpoint_A = load_checkpoint_fields(
        args.ss_filename
    )
    comm = mesh.comm
    validate_cfg_and_mesh(cfg, mesh)

    if not args.no_grid and comm.size != 1:
        raise RuntimeError(
            "NPZ export requires one MPI rank. Use --no-grid for a parallel "
            "checkpoint-only run."
        )

    A_field, A_source = make_A_field(
        mesh, Q, checkpoint_A, cfg, explicit_keys
    )

    dt = args.dt
    if dt is None:
        dt = float(
            cfg.get(
                "run_dt",
                cfg.get("fine_dt", cfg.get("coarse_dt", 1.0)),
            )
        )

    output_checkpoint, output_config, grid_output = derive_output_paths(args)

    print0(comm, f"Loaded spin-up cfg from: {cfg_source}")
    print0(comm, f"A field source: {A_source}")
    print0(
        comm,
        "Forward parameters: "
        f"C={float(cfg['C'])}, A(base)={cfg.get('A', '<saved field>')}, "
        f"Ax={float(cfg.get('Ax', 0.0))}, Ay={float(cfg.get('Ay', 0.0))}, "
        f"a={float(cfg['a'])}, omega={float(cfg.get('omega', 0.0))}, "
        f"duration={args.final_time}, dt={dt}",
    )
    if changes:
        print0(comm, "Explicit cfg overrides:")
        for change in changes:
            print0(
                comm,
                f"  {change['key']}: {change['old']!r} -> "
                f"{change['new']!r} ({change['source']})",
            )
    else:
        print0(comm, "No cfg overrides supplied; using the spin-up settings.")

    model = icepack.models.IceStream(friction=friction)
    primary_solver = make_solver(
        "petsc", model, monitor=bool(cfg.get("monitor_snes", False))
    )
    fallback_solver = make_solver("icepack", model, monitor=False)

    initial_fields, final_fields, time_steps = run_simulation(
        primary_solver,
        fallback_solver,
        fields,
        bed=bed,
        A_field=A_field,
        C_value=float(cfg["C"]),
        accumulation_value=float(cfg["a"]),
        omega=float(cfg.get("omega", 0.0)),
        duration=float(args.final_time),
        dt=float(dt),
        print_every=int(cfg.get("step_print_every", 10)),
    )

    run_cfg = dict(cfg)
    run_cfg.update(
        {
            "run_dt": float(dt),
            "run_duration": float(args.final_time),
            "run_time_steps": [float(value) for value in time_steps],
            "parent_checkpoint": os.path.abspath(args.ss_filename),
            "parent_cfg_source": cfg_source,
            "A_source": A_source,
            "A_field_saved_value": (
                float(cfg["A"])
                if A_source == "overridden-constant"
                else (
                    cfg.get("A_field_saved_value", cfg.get("A"))
                    if A_source in {"checkpoint", "cfg-constant-fallback"}
                    else None
                )
            ),
            "forward_overrides": changes,
        }
    )
    if not args.no_checkpoint:
        run_cfg["output_stem"] = os.fspath(output_checkpoint.with_suffix(""))

    write_json(run_cfg, os.fspath(output_config), comm)
    print0(comm, f"Saved effective cfg: {output_config}")

    if not args.no_checkpoint:
        save_checkpoint(
            os.fspath(output_checkpoint),
            mesh,
            Q,
            V,
            final_fields,
            bed,
            A_field,
            run_cfg,
            os.fspath(output_config),
        )

    if not args.no_grid:
        save_grid_npz(
            os.fspath(grid_output),
            mesh,
            Q,
            initial_fields,
            final_fields,
            bed,
            A_field,
            run_cfg,
        )


if __name__ == "__main__":
    main()
